# 06｜VLM 上的 LoRA / QLoRA 多模態微調

> 本 notebook 是 `05-Multimodal` 模組的終點，也是整套 cookbook「單模態微調 → 多模態微調」學習弧線的閉合點。

## 學習目標

讀完這份 notebook，你會能夠：

1. 把 `03-PEFT` 的 LoRA 與 `04-kbits-tuning` 的 QLoRA / `BitsAndBytesConfig` **原封不動地遷移到視覺語言模型（VLM）**，並清楚說出三個關鍵差異。
2. 用 `AutoProcessor`（而非 `tokenizer`）處理「影像 + 中文 instruction + response」圖文對。
3. 寫出多模態 collator，正確產出 `pixel_values + input_ids + labels`。
4. 套用**標籤遮罩**：把 image token 與 prompt 設成 `-100`，只在 response 上算 loss。
5. 設定 VLM 專屬的 `target_modules`（LLM 的 `q/k/v/o` + 視覺 projector），並凍結 vision encoder。
6. 用 `SFTTrainer(peft_config=...)` 完成 4-bit QLoRA 訓練、做訓練前後的 VQA 對照、判斷 `merge_and_unload()` vs adapter-only 部署的取捨，並 `push_to_hub` 附多模態 model card。

## 前置知識（請先完成）

本 notebook **假設你已經做過**下列單模態版本，不再重述其原理：

- LoRA 基礎：[`../../03-PEFT/01-LoRA/chatbot_lora.ipynb`](../../03-PEFT/01-LoRA/chatbot_lora.ipynb)（`LoraConfig` / `get_peft_model` / `target_modules`）
- QLoRA 與 4-bit 量化：[`../../04-kbits-tuning/04-4bits_training/llama2_qlora_4bit.ipynb`](../../04-kbits-tuning/04-4bits_training/llama2_qlora_4bit.ipynb)（`BitsAndBytesConfig` / `prepare_model_for_kbit_training` / `-100` 遮罩）
- 量化原理與 VRAM 估算：[`../../04-kbits-tuning/README.md`](../../04-kbits-tuning/README.md)
- VLM 載入與 chat 模板：[`../03-vlm_vqa_captioning/vlm_vqa_captioning.ipynb`](../03-vlm_vqa_captioning/vlm_vqa_captioning.ipynb)（`apply_chat_template` 承載影像）
- 純文字 chat 模板：[`../../02-Adv-tasks/08-generative_chatbot/chatbot.ipynb`](../../02-Adv-tasks/08-generative_chatbot/chatbot.ipynb)

## 與既有模組的銜接

| 你在前面學過的 | 在這份 notebook 變成 |
| :--- | :--- |
| `tokenizer` 把字串轉 `input_ids` | `AutoProcessor` 同時處理影像與文字，產出 `pixel_values + input_ids` |
| `target_modules=["q_proj","v_proj",...]` | 多加上視覺 projector，並凍結 vision tower |
| prompt 區段設 `-100` 只在回應算 loss | image token **也**要設 `-100`（延伸到影像模態） |
| `Trainer` / `SFTTrainer` + `DataCollatorForSeq2Seq` | 同一個 `SFTTrainer`，換成多模態 collator |

一句話總結本 notebook 的精神：**多模態微調不是另起爐灶，95% 的流程你在 03/04 已經做過，真正新的只有「影像怎麼進序列、image token 怎麼遮罩、用 processor 不用 tokenizer」這三件事。**

## 0. 版本鎖定（2026）

**WHY 放最前面**：多模態微調牽涉 `transformers` / `trl` / `peft` / `bitsandbytes` 四個套件的交互相容性，版本飄移是最常見的「明明照抄卻跑不起來」來源。鎖定版本與全 repo 一致（見 [`../README.md`](../README.md) 第 5 節），不另開一套。

下方 cell 預設**註解掉**，因為在已配置好的環境重跑會浪費時間；首次執行請取消註解。

In [ ]:
# Run once in a fresh environment. Versions locked to the repo-wide 2026 baseline.
# Qwen2.5-VL needs qwen-vl-utils for its image/video preprocessing helpers.
# !pip install -q \
#     "transformers>=4.46" \
#     "datasets>=3.0" \
#     "trl>=0.12" \
#     "peft>=0.13" \
#     "accelerate>=1.0" \
#     "bitsandbytes>=0.44" \
#     "evaluate>=0.4" \
#     "qwen-vl-utils" \
#     "pillow" "torchvision"

import transformers, datasets, trl, peft, bitsandbytes, torch
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("trl         ", trl.__version__)
print("peft        ", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)
print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. 回顧 03/04，並總覽「遷移到 VLM」的差異

**WHY 先回顧**：與其背一套新 API，不如先把你已經會的東西攤開來看，再標出哪幾格需要改。下面這個 cell 不做任何訓練，只是把單模態與多模態的對照寫成可印出的對照表，當作整份 notebook 的地圖。

核心訊息：流程骨架完全一致 —— **量化載入 → `prepare_model_for_kbit_training` → `LoraConfig` → `SFTTrainer` → 存檔 / 推論**。真正改動的只有右欄標星號的三處。

In [ ]:
# A side-by-side map: what stays the same vs. what changes when moving to a VLM.
# This is documentation-as-code; nothing here trains.
from textwrap import dedent

comparison = dedent("""\
    step                     | single-modal (03/04)              | VLM (this notebook)
    -------------------------|-----------------------------------|-------------------------------------------
    preprocessing            | AutoTokenizer -> input_ids        | * AutoProcessor -> pixel_values + input_ids
    4-bit load               | BitsAndBytesConfig(load_in_4bit)  | same BitsAndBytesConfig
    kbit prep                | prepare_model_for_kbit_training() | same
    LoRA targets             | q/k/v/o proj                      | * q/k/v/o proj + vision projector
    frozen parts             | (base weights frozen by 4-bit)    | * also freeze the vision encoder
    label masking            | prompt -> -100, loss on response  | * image tokens + prompt -> -100
    collator                 | DataCollatorForSeq2Seq            | * custom multimodal collator
    trainer                  | SFTTrainer(peft_config=...)       | same SFTTrainer, multimodal collator
    save / deploy            | save_pretrained / merge_and_unload| same
""")
print(comparison)
print("Lines marked with * are the ONLY genuinely new things in VLM fine-tuning.")

### 1.1 選定模型與輕量替代

**WHY 先定模型常數**：不同 VLM 的 `target_modules` 名稱、projector 模組名、image token 數量都不一樣（這是 VLM 微調最容易踩雷之處）。把模型相關常數集中在一個 cell，之後切換模型只改這裡。

本 notebook 以 **`Qwen/Qwen2.5-VL-7B-Instruct`**（4-bit QLoRA、中文最強）為主線，並列出對照模型：

| 模型 | 定位 | 概略訓練 VRAM（4-bit QLoRA） |
| :--- | :--- | :--- |
| `Qwen/Qwen2.5-VL-7B-Instruct` | 2026 首選，中文強 | 約 12–16 GB |
| `llava-hf/llava-1.5-7b-hf` | 經典對照，社群資源多 | 約 10–14 GB |
| `HuggingFaceTB/SmolVLM-Instruct` | **低 VRAM 對照**，< 8 GB GPU 首選 | 約 5–8 GB |
| `google/paligemma2-3b-pt-224` | 輕量對照，需先接受授權 | 約 6–10 GB |

> 警告：7B VLM 的 4-bit QLoRA 訓練實際吃約 12–16 GB VRAM（含梯度、optimizer state、activation）。若 GPU < 12 GB，把 `MODEL_ID` 換成 `HuggingFaceTB/SmolVLM-Instruct`，本 notebook 其餘程式碼幾乎不用改 —— 這正是 processor 抽象帶來的可攜性。

In [ ]:
import torch

# --- Primary model (Qwen2.5-VL, best Chinese). Switch the comments to swap. ---
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
# MODEL_ID = "llava-hf/llava-1.5-7b-hf"          # classic baseline
# MODEL_ID = "HuggingFaceTB/SmolVLM-Instruct"   # low-VRAM (<8GB) fallback
# MODEL_ID = "google/paligemma2-3b-pt-224"      # lightweight, needs license accept

SEED = 42
OUTPUT_DIR = "./qwen25vl-vqa-qlora"
HUB_REPO_ID = "your-username/qwen25vl-vqa-qlora-zh"  # change before push_to_hub

# Different VLMs name their LoRA targets differently. We resolve this dynamically
# later via a regex, but it is useful to know the families up front:
#   Qwen2.5-VL : LLM attn = q_proj/k_proj/v_proj/o_proj, projector = "merger"
#   LLaVA-1.5  : LLM attn = q_proj/k_proj/v_proj/o_proj, projector = "multi_modal_projector"
#   SmolVLM    : LLM attn = q_proj/k_proj/v_proj/o_proj, projector = "connector"
print("Selected model:", MODEL_ID)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu (training will be impractically slow)")

## 2. 載入小型多模態指令資料集

**WHY 用小資料集**：教學目的是讓你看懂「圖文對 → 遮罩 → loss」的資料流，不是刷 benchmark。我們取一個帶影像的中文 VQA / 指令資料子集，控制在數百筆，讓整條管線能在單張消費級 GPU 上幾分鐘內跑完一個 epoch。

**這一步在單模態版本你已做過**：在 [`../../04-kbits-tuning/04-4bits_training/llama2_qlora_4bit.ipynb`](../../04-kbits-tuning/04-4bits_training/llama2_qlora_4bit.ipynb) 你用 `load_dataset` 載入 alpaca 中文指令集。唯一新增的欄位是 `image`（一個 `PIL.Image`）。

資料每筆需要三個概念欄位：`image`（圖）、`question`（中文 instruction）、`answer`（期望 response）。下面示範用一個公開的小型 VQA 資料集；若你想用自備資料，看 2.2 的 schema 即可。

In [ ]:
from datasets import load_dataset

# A small, image-bearing instruction dataset. We keep only a few hundred rows so the
# whole pipeline runs quickly. Any dataset with an image + a Q/A text pair works.
# Here we use a compact VQA-style set; swap for a LLaVA-Instruct subset if you prefer.
raw = load_dataset("merve/vqav2-small", split="validation")

# Keep it tiny for teaching. In a real run you would use far more data.
raw = raw.shuffle(seed=SEED).select(range(300))

# Normalize to a consistent schema: image / question / answer.
# (Field names differ across datasets; this is the only dataset-specific glue.)
def to_schema(ex):
    # vqav2-small exposes "multiple_choice_answer" as the gold short answer.
    return {
        "image": ex["image"],
        "question": ex["question"],
        "answer": str(ex["multiple_choice_answer"]),
    }

ds = raw.map(to_schema, remove_columns=[c for c in raw.column_names if c not in ("image",)])
ds = ds.train_test_split(test_size=0.1, seed=SEED)
print(ds)
print("sample question:", ds["train"][0]["question"])
print("sample answer  :", ds["train"][0]["answer"])
ds["train"][0]["image"]

### 2.1 把英文 VQA 轉成中文指令風格（可選）

**WHY**：本 cookbook 主打中文教學，且 inventory 強調「中文 instruction + response」。`vqav2-small` 是英文的，這裡示範一個輕量包裝：用中文模板把 `question` 重新表述成中文指令，answer 維持簡短。實務上你會直接用中文資料集（如 LLaVA-Instruct 的中文版或自建集），這步只是讓教學示例貼近目標語言。

> 注意：這只是「指令外殼中文化」，不是翻譯答案內容。真正的中文 VLM 微調請用真正的中文圖文資料，這裡重點仍是資料流而非語料品質。

In [ ]:
# Wrap each English question in a Chinese instruction shell, keep the short answer.
# This is purely to make the teaching example read in zh-TW; not a translation step.
ZH_TEMPLATE = "請仔細觀察這張圖片並用簡短的中文回答下列問題：{q}"

def zh_wrap(ex):
    ex["question"] = ZH_TEMPLATE.format(q=ex["question"])
    return ex

ds = ds.map(zh_wrap)
print(ds["train"][0]["question"])
print("answer:", ds["train"][0]["answer"])

### 2.2 自備資料的 schema

**WHY 寫清楚**：學員最常問「我自己的圖怎麼餵」。只要你的資料能整成下列三欄，後面所有程式碼都不用改：

```python
# 自備資料只需符合此 schema：
# {
#   "image":    PIL.Image.Image,   # 或一個能被 PIL 開啟的路徑/位元組
#   "question": str,               # 中文 instruction
#   "answer":   str,               # 期望的 response
# }
# 例如從 jsonl + 影像資料夾建：
# from datasets import Dataset
# from PIL import Image
# rows = [{"image": Image.open(p).convert("RGB"), "question": q, "answer": a} for p, q, a in ...]
# ds = Dataset.from_list(rows)
```

關鍵：影像務必 `convert("RGB")`，避免灰階/RGBA 通道數不一致讓 `pixel_values` 形狀對不上。

## 3. AutoProcessor 處理圖文對

**WHY 用 processor 而非 tokenizer**：這是多模態與單模態最根本的差異。`AutoProcessor` 同時做兩件事：(1) 用 `image_processor` 把 `PIL.Image` 轉成 `pixel_values`（resize / normalize 到模型訓練時的 mean/std）；(2) 用內建 tokenizer 把文字轉成 `input_ids`，並**在序列裡正確插入 image placeholder token**。

手動拼 `pixel_values` 與 `input_ids` 幾乎必然踩到「image token 數量對不上 patch 數量」的坑 —— 這是 VLM 微調第一大雷。讓 processor 一手包辦就不會錯。

**這一步在單模態版本你已做過**：等同於 04 的 `tokenizer(...)`，只是現在多吃一個 `images=` 參數。

In [ ]:
from transformers import AutoProcessor

# trust_remote_code is needed for some VLM processors (e.g. Qwen2.5-VL helpers).
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

# A VLM processor bundles a tokenizer + an image_processor under one roof.
print(type(processor).__name__)
print("has tokenizer      :", hasattr(processor, "tokenizer"))
print("has image_processor:", hasattr(processor, "image_processor"))

# Training needs right padding for the LLM head; set it explicitly to avoid
# the silent left-pad default that some chat processors ship with.
if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "right"

### 3.1 用 chat 模板把一筆樣本變成訓練序列

**WHY 用 `apply_chat_template`**：VLM 的 prompt 不是裸字串，而是帶角色的 messages，且影像以 `{"type": "image"}` 的結構嵌入 content。`processor.apply_chat_template` 會把這個結構展開成模型認得的特殊 token 序列（含 image placeholder），確保**訓練時的格式與推論時完全一致** —— 不一致是 VLM 微調第二大雷（訓練學到的格式推論時對不上，效果歸零）。

**這一步在單模態版本你已做過**：[`../../02-Adv-tasks/08-generative_chatbot/chatbot.ipynb`](../../02-Adv-tasks/08-generative_chatbot/chatbot.ipynb) 的 `apply_chat_template`。差別只是 content 從字串變成 `[{"type":"image"},{"type":"text",...}]` 的 list。

In [ ]:
def build_messages(question: str, answer: str | None = None):
    """Build a chat-format conversation. answer=None => inference prompt only."""
    # The user turn carries BOTH the image placeholder and the text question.
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},                       # image placeholder token(s)
                {"type": "text", "text": question},
            ],
        }
    ]
    if answer is not None:
        # During training we append the assistant turn so it becomes part of the
        # target sequence (and the part we will actually compute loss on).
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages

# Inspect the rendered text template (image token is just a special marker in the string).
sample = ds["train"][0]
rendered = processor.apply_chat_template(
    build_messages(sample["question"], sample["answer"]),
    tokenize=False,
)
print(rendered)

## 4. 標籤遮罩策略：image token 與 prompt 設 -100，只在 response 算 loss

**WHY 這是整份 notebook 最該理解的觀念**：因果語言模型在每個位置預測下一個 token。但我們**不想讓模型學著去「生成 prompt」或「生成 image placeholder」** —— 那既無意義又會稀釋學習訊號。所以我們把不該算 loss 的位置標成 `-100`（PyTorch CrossEntropy 的 `ignore_index`），只在 assistant 的回應 token 上算 loss。

**這一步在單模態版本你已做過**：04 的 QLoRA 就是把 prompt 段設 `-100`、只在回應算 loss。

**多模態的新增點**：image placeholder token 也必須設 `-100`。它們是「視覺輸入的佔位」，不是要被「生成」的文字目標；若不遮，loss 會試圖把這些位置當文字預測，訓練直接壞掉。

下面的 collator 用一個乾淨的做法消除特殊情況：**先對整段（prompt+answer）算一份 `labels = input_ids.clone()`，再把「非 answer 區段」整片設 -100**。我們透過「只渲染 prompt 部分的 token 長度」來定位 answer 的起點，這樣 image token 自然落在 prompt 區段裡、一併被遮掉，不需要額外去逐一找 image token id。

In [ ]:
import torch

IGNORE_INDEX = -100

def collate_fn(examples):
    """Multimodal collator: produces pixel_values + input_ids + masked labels.

    Masking rule (the whole point of this cell):
      - everything in the prompt segment (system/user text AND image tokens) -> -100
      - only the assistant answer tokens keep their real ids -> loss computed here
    """
    images = [ex["image"].convert("RGB") for ex in examples]

    # Full text = prompt + assistant answer (this is what the model is trained on).
    full_texts = [
        processor.apply_chat_template(
            build_messages(ex["question"], ex["answer"]), tokenize=False
        )
        for ex in examples
    ]
    # Prompt-only text = up to (and including) the generation prompt, no answer.
    # Its token length tells us where the answer starts, so we can mask everything
    # before it -- including image placeholder tokens, which live in the prompt.
    prompt_texts = [
        processor.apply_chat_template(
            build_messages(ex["question"], answer=None),
            tokenize=False,
            add_generation_prompt=True,
        )
        for ex in examples
    ]

    # One processor call handles BOTH text tokenization and image -> pixel_values,
    # guaranteeing the image-token count matches the patch count.
    batch = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    labels = batch["input_ids"].clone()

    # Mask padding so it never contributes to loss.
    pad_id = processor.tokenizer.pad_token_id
    labels[labels == pad_id] = IGNORE_INDEX

    # Mask the entire prompt segment (text prompt + image tokens) per example.
    for i, prompt_text in enumerate(prompt_texts):
        prompt_len = len(
            processor.tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        )
        labels[i, :prompt_len] = IGNORE_INDEX

    batch["labels"] = labels
    return batch

# Smoke-test the collator on 2 rows and confirm the masking shape.
_demo = collate_fn([ds["train"][0], ds["train"][1]])
print("keys        :", list(_demo.keys()))
print("input_ids   :", tuple(_demo["input_ids"].shape))
print("pixel_values:", tuple(_demo["pixel_values"].shape))
print("labels      :", tuple(_demo["labels"].shape))
print("masked tokens (-100):", int((_demo["labels"] == IGNORE_INDEX).sum()),
      "/ kept:", int((_demo["labels"] != IGNORE_INDEX).sum()))

### 4.1 視覺化驗證遮罩是否正確

**WHY 一定要驗**：標籤遮罩錯了，loss 看起來照樣會降，但模型學的是錯的東西，等到推論才發現為時已晚。花十秒把「保留下來算 loss 的 token」解碼出來，肉眼確認它**只包含 answer、不含 prompt 也不含 image token**，是最划算的除錯。

In [ ]:
# Decode only the kept (non -100) tokens of the first example. It should read like
# the assistant answer, NOT the question and NOT any image placeholder noise.
labels0 = _demo["labels"][0]
kept_ids = labels0[labels0 != IGNORE_INDEX]
print("kept (loss is computed on these tokens):")
print(repr(processor.tokenizer.decode(kept_ids)))
print()
print("expected answer:", ds["train"][0]["answer"])

## 5. 4-bit 量化載入 + prepare_model_for_kbit_training()

**WHY 完全沿用 04 的 QLoRA 配方**：7B VLM 的全精度權重約 14–15 GB，4-bit NF4 量化把基底壓到約 4–5 GB，才放得進消費級 GPU。`BitsAndBytesConfig` 的四個參數與 [`../../04-kbits-tuning/04-4bits_training/llama2_qlora_4bit.ipynb`](../../04-kbits-tuning/04-4bits_training/llama2_qlora_4bit.ipynb) 一模一樣 —— 這就是知識遷移的證據：**量化配方不分模態**。

`prepare_model_for_kbit_training()` 做三件事：把 layernorm 升回 fp32 穩定訓練、開啟梯度檢查點所需的 input require_grad、關閉 use_cache。也與 04 相同。

> 注意：VLM 用對應的 `AutoModelForImageTextToText`（或模型專屬類別）載入，而非 `AutoModelForCausalLM`。這是少數模態相關的 API 差異。

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForImageTextToText

# Identical QLoRA quantization recipe as the text-only notebook (04). Modality-agnostic.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                 # NF4 > plain int4 for LLM/VLM weights
    bnb_4bit_compute_dtype=torch.bfloat16,    # matmuls run in bf16
    bnb_4bit_use_double_quant=True,           # quantize the quant constants too
)

# VLMs use the image-text-to-text auto class, not AutoModelForCausalLM.
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# WARNING: this download + load needs ~12-16 GB VRAM for a 7B VLM during training.
# If you OOM, switch MODEL_ID to HuggingFaceTB/SmolVLM-Instruct in cell 1.1.
print("loaded:", type(model).__name__)

In [ ]:
from peft import prepare_model_for_kbit_training

# Same kbit prep as 04: upcasts norms to fp32, disables use_cache, enables
# input grads for gradient checkpointing. Modality makes no difference here.
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True
)
print("model prepared for k-bit training")

## 6. LoraConfig 多模態 target_modules + 凍結 vision encoder

**WHY 這是 VLM LoRA 的核心差異**：純文字 LoRA 只把適配器掛在 LLM 的注意力投影（`q/k/v/o`）。VLM 多了一條視覺路徑：`vision encoder → projector → LLM`。

- **LLM 的 `q/k/v/o`**：要掛 LoRA（與 03/04 相同），讓語言端適配新任務。
- **視覺 projector**：建議掛 LoRA。它是「把視覺特徵翻譯成 LLM 語言」的橋樑，微調它能讓模型更好地對齊你的圖文任務。
- **vision encoder（vision tower）**：通常**凍結**。它已在海量影像上預訓練，再動容易破壞通用視覺表徵且成本高。凍結是預設、也是最省 VRAM 的選擇。

**這一步在單模態版本你已做過**：03 的 `LoraConfig` + `get_peft_model`。差別只在 `target_modules` 多了 projector，以及多一道「確認 vision tower 凍結」。

下面用一個小工具**自動偵測**哪些 Linear 層屬於 LLM attention 與 projector、哪些屬於 vision tower，避免硬編模組名（不同 VLM 命名不同，硬編是第三大雷）。

In [ ]:
import re
import torch.nn as nn

# Auto-discover candidate LoRA target module names instead of hard-coding them
# (Qwen calls its projector "merger", LLaVA "multi_modal_projector", SmolVLM "connector").
# Rule: take Linear layers in the LLM attention block + the projector, but NOT the
# vision encoder/tower.
VISION_KEYWORDS = ("vision_tower", "vision_model", "visual.blocks", "visual.patch")
LLM_ATTN = ("q_proj", "k_proj", "v_proj", "o_proj")
PROJECTOR_KEYWORDS = ("projector", "merger", "connector", "mm_proj")

target_modules = set()
for name, module in model.named_modules():
    if not isinstance(module, (nn.Linear,)) and "Linear4bit" not in type(module).__name__:
        continue
    in_vision = any(k in name for k in VISION_KEYWORDS)
    if in_vision:
        continue  # keep the vision encoder frozen -> never a LoRA target
    leaf = name.split(".")[-1]
    if leaf in LLM_ATTN or any(k in name for k in PROJECTOR_KEYWORDS):
        target_modules.add(leaf)

target_modules = sorted(target_modules)
print("resolved target_modules:", target_modules)
# Compare to the text-only baseline you used in 04:
print("text-only baseline was :", ["q_proj", "k_proj", "v_proj", "o_proj"])

In [ ]:
# Explicitly freeze the vision tower. With 4-bit + LoRA the base is already frozen,
# but this makes the intent unambiguous and guards against accidental updates.
frozen, trainable_pre = 0, 0
for name, param in model.named_parameters():
    if any(k in name for k in VISION_KEYWORDS):
        param.requires_grad = False
        frozen += param.numel()
print(f"vision-tower params explicitly frozen: {frozen/1e6:.1f}M")

In [ ]:
from peft import LoraConfig, get_peft_model

# Same LoraConfig shape as 03/04; only target_modules differs (now includes projector).
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",          # the VLM is a causal LM under the hood
    target_modules=target_modules,  # LLM q/k/v/o + projector, vision tower excluded
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # should be a tiny % of total -- the LoRA win

## 7. SFTTrainer + SFTConfig 訓練

**WHY 用 `SFTTrainer` 而非裸 `Trainer`**：`trl` 的 `SFTTrainer` 原生支援 `peft_config` 與自訂多模態 `data_collator`，把 LoRA 注入、packing、評估迴圈都包好了。我們已經手動 `get_peft_model`，所以這裡直接把 PEFT 模型交給它，並餵入第 4 步的多模態 collator。

**這一步在單模態版本你已做過**：04 的 `SFTTrainer` / `SFTConfig`。設定值（`bf16`、`gradient_checkpointing`、`save_safetensors`、`seed=42`、`warmup_ratio=0.1`、`lr_scheduler_type="cosine"`）與全 repo 慣例一致 —— 又一個「流程不變」的證據。

關鍵差異只有一行：`remove_unused_columns=False`。預設 `SFTTrainer` 會丟掉它不認得的欄位，但我們的 `image` 欄位要進 collator，必須保留。這是多模態訓練最常被忽略的設定。

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,                 # teaching run; raise for real training
    per_device_train_batch_size=1,      # VLMs are memory-heavy; keep batch small
    gradient_accumulation_steps=8,      # effective batch = 1 * 8
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    save_strategy="epoch",
    save_safetensors=True,              # repo-wide convention
    seed=SEED,
    report_to="none",
    # --- the one multimodal-specific must-have ---
    remove_unused_columns=False,        # keep the 'image' column for the collator
    dataset_kwargs={"skip_prepare_dataset": True},  # we pre-format via our collator
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=collate_fn,          # our multimodal collator from step 4
)
print("trainer ready")

### 7.1 開始訓練

**WHY 先存基線輸出再訓練**：第 8 步要做訓練前後對照，所以下一個 cell 先別急著跑訓練 —— 我們在第 8 步先抓「訓練前」的 VQA 輸出，再回來執行 `trainer.train()`。這裡先放訓練呼叫，但建議的執行順序是：先跑第 8 步的「Before」，再回到這裡 `train()`，最後跑第 8 步的「After」。

> 警告：7B VLM 即使 1 個 epoch、300 筆，在單張 24GB GPU 上約需數分鐘到十幾分鐘；低階 GPU 請改用 SmolVLM 並縮小資料。

In [ ]:
# Run AFTER capturing the "before" output in step 8.
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)   # saves the LoRA adapter (safetensors)
print(train_result.metrics)

## 8. 訓練前後 VQA 輸出對照（定性評估）

**WHY 定性對照**：VLM 微調的成效往往不是單一數字能說清的，最直觀的驗證是拿同一張圖、同一個問題，看微調前後回答的差異（語氣是否更貼近訓練資料、答案是否更簡潔正確）。

**這一步在單模態版本你已做過**：04 的 `model.generate` 推論。差別只在輸入要用 `processor.apply_chat_template` 帶影像，且 **`add_generation_prompt=True`** —— 推論模板必須與訓練時的格式一致，否則微調白做（這點在第 9 步會再強調）。

下面的 `vqa_generate` 是訓練前後共用的推論函式。

In [ ]:
@torch.no_grad()
def vqa_generate(vlm, image, question, max_new_tokens=64):
    """Inference must mirror training: same chat template, add_generation_prompt=True."""
    messages = build_messages(question, answer=None)
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,   # ask the model to start the assistant turn
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    # apply_chat_template handles text tokens; attach the actual pixel_values.
    pixel = processor(images=image.convert("RGB"), text="", return_tensors="pt")["pixel_values"]
    inputs = {k: v.to(vlm.device) for k, v in inputs.items()}
    inputs["pixel_values"] = pixel.to(vlm.device, dtype=torch.bfloat16)

    out = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    # Strip the prompt portion; decode only the newly generated answer.
    gen = out[0][inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(gen, skip_special_tokens=True).strip()

# Pick a fixed eval example to compare before vs. after.
eval_ex = ds["test"][0]
eval_ex["image"]

In [ ]:
# ---- BEFORE training ----
# Run this cell BEFORE executing trainer.train() (cell in 7.1).
before = vqa_generate(model, eval_ex["image"], eval_ex["question"])
print("Q:", eval_ex["question"])
print("gold:", eval_ex["answer"])
print("BEFORE:", before)

In [ ]:
# ---- AFTER training ----
# Run this cell AFTER trainer.train() has finished.
after = vqa_generate(model, eval_ex["image"], eval_ex["question"])
print("Q:", eval_ex["question"])
print("gold :", eval_ex["answer"])
print("AFTER:", after)

# Compare a few more for a fuller qualitative picture.
for ex in ds["test"].select(range(min(3, len(ds["test"])))):
    print("-" * 60)
    print("Q   :", ex["question"])
    print("gold:", ex["answer"])
    print("pred:", vqa_generate(model, ex["image"], ex["question"]))

## 9. merge_and_unload() vs adapter-only 部署，與推論模板一致性

**WHY 要做這個決策**：訓練產出的是 LoRA adapter（通常只有幾 MB ~ 幾十 MB）。部署時有兩條路，取捨如下：

| 方案 | 做法 | 優點 | 缺點 |
| :--- | :--- | :--- | :--- |
| **adapter-only** | 部署時載入 base + adapter | 檔案小、可熱插拔多個任務 adapter、base 可共用 | 推論多一層 adapter 計算、需確保 base 版本一致 |
| **merge_and_unload()** | 把 adapter 權重合進 base，輸出單一模型 | 推論零額外開銷、部署單純 | 失去可插拔性、合併後若用 4-bit base 會有精度損失（建議在 fp16/bf16 base 上合併） |

**重要陷阱**：在 4-bit 量化的 base 上直接 `merge_and_unload()` 會把 LoRA 合進去量化權重，精度會掉。正確做法是**重新以 bf16 載入 base，套上 adapter，再 merge**。下面示範兩條路。

**這一步在單模態版本你已做過**：03 的 `lora_inference.ipynb` 已示範 adapter 載入；VLM 完全相同，唯一要再三強調的是 —— **推論的 chat 模板與 `add_generation_prompt=True` 必須與訓練時一致**，否則模型在「沒看過的格式」上推論，微調效果會消失。

In [ ]:
# --- Option A: adapter-only deployment (recommended default) ---
# Save just the LoRA adapter; tiny and hot-swappable.
ADAPTER_DIR = OUTPUT_DIR + "-adapter"
model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
processor.save_pretrained(ADAPTER_DIR)  # ALWAYS ship the processor with the adapter
print("adapter + processor saved to", ADAPTER_DIR)

# Re-load for serving:
# from peft import PeftModel
# base = AutoModelForImageTextToText.from_pretrained(
#     MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)
# served = PeftModel.from_pretrained(base, ADAPTER_DIR)
# processor = AutoProcessor.from_pretrained(ADAPTER_DIR, trust_remote_code=True)

In [ ]:
# --- Option B: merge_and_unload for a single self-contained model ---
# Merge on a bf16 base (NOT the 4-bit one) to avoid precision loss.
# from peft import PeftModel
# bf16_base = AutoModelForImageTextToText.from_pretrained(
#     MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)
# merged = PeftModel.from_pretrained(bf16_base, ADAPTER_DIR)
# merged = merged.merge_and_unload()                 # fold LoRA into base weights
# MERGED_DIR = OUTPUT_DIR + "-merged"
# merged.save_pretrained(MERGED_DIR, safe_serialization=True)
# processor.save_pretrained(MERGED_DIR)
# print("merged full model saved to", MERGED_DIR)
print("Option B is commented out; uncomment to produce a merged model.")

## 10. push_to_hub + 多模態 model card，與成本/下一步討論

**WHY 推上 Hub**：分享、版本控制、可重現。與全 repo 慣例一致，訓練 notebook 結尾示範 `push_to_hub` 並附 model card。

**這一步在單模態版本你已做過**：04 的 `push_to_hub`。多模態的差別只在 model card 要寫清楚「這是 VLM adapter、需要哪個 base、推論時的 chat 模板格式」。

In [ ]:
# Push the adapter (small) + processor + model card. Uncomment to run.
# Requires: huggingface-cli login  (or HF_TOKEN env var)

MODEL_CARD = f"""---
library_name: peft
base_model: {MODEL_ID}
pipeline_tag: image-text-to-text
tags:
  - vlm
  - lora
  - qlora
  - multimodal
  - zh
language:
  - zh
---

# Qwen2.5-VL VQA QLoRA (zh-TW)

LoRA/QLoRA adapter fine-tuned on a small Chinese VQA instruction set.

- **Base model**: `{MODEL_ID}` (load in 4-bit NF4 for low VRAM)
- **Trained modules**: LLM q/k/v/o projections + vision projector (vision encoder frozen)
- **Label masking**: image tokens + prompt set to -100, loss on the answer only

## Inference (must match training format)

```python
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel
import torch

base = AutoModelForImageTextToText.from_pretrained(
    "{MODEL_ID}", device_map="auto", torch_dtype=torch.bfloat16, trust_remote_code=True)
model = PeftModel.from_pretrained(base, "{HUB_REPO_ID}")
processor = AutoProcessor.from_pretrained("{HUB_REPO_ID}", trust_remote_code=True)

messages = [{{"role": "user", "content": [
    {{"type": "image"}}, {{"type": "text", "text": "這張圖是什麼?"}}]}}]
# build inputs with apply_chat_template(add_generation_prompt=True) -> model.generate(...)
```
"""

# import os
# os.makedirs(ADAPTER_DIR, exist_ok=True)
# with open(os.path.join(ADAPTER_DIR, "README.md"), "w", encoding="utf-8") as f:
#     f.write(MODEL_CARD)
# model.push_to_hub(HUB_REPO_ID)
# processor.push_to_hub(HUB_REPO_ID)
# print("pushed to", HUB_REPO_ID)
print(MODEL_CARD[:400], "...")

### 10.1 VRAM / 時間成本與對照

**WHY 談成本**：選模型與選量化檔位的決策，本質是 VRAM 與品質的取捨。把這份對照當作你下次規劃微調的 checklist（估算原理見 [`../../04-kbits-tuning/README.md`](../../04-kbits-tuning/README.md)）。

| 模型 | 量化 | 訓練 VRAM（概略） | 1 epoch / 300 筆（單張 24GB） | 適用情境 |
| :--- | :--- | :--- | :--- | :--- |
| Qwen2.5-VL-7B | 4-bit QLoRA | 12–16 GB | 數分鐘 | 中文最佳，主線選擇 |
| LLaVA-1.5-7B | 4-bit QLoRA | 10–14 GB | 數分鐘 | 英文社群資源多 |
| SmolVLM-Instruct | 4-bit QLoRA | 5–8 GB | 1–3 分鐘 | < 8GB GPU、原型迭代 |
| paligemma2-3b | 4-bit QLoRA | 6–10 GB | 數分鐘 | 輕量、需接受授權 |

降 VRAM 的旋鈕（與 04 相同）：縮小 `r`、減少 `target_modules`、降 `max_new_tokens` / 序列長度、提高 `gradient_accumulation_steps` 以維持有效 batch。

## 小結

你完成了整套 cookbook「單模態微調 → 多模態微調」學習弧線的閉合。回顧本 notebook，真正屬於多模態的新知識只有三件事，其餘全是 03/04 的再應用：

1. **用 `AutoProcessor` 而非 `tokenizer`**：一手包辦 `pixel_values + input_ids`，避免 image token 數量對不上。
2. **`target_modules` 含 LLM 的 `q/k/v/o` + 視覺 projector，凍結 vision encoder**：視覺路徑只動橋樑（projector），不動已預訓練好的視覺編碼器。
3. **標籤遮罩擴展到 image token**：image placeholder 與 prompt 一律 `-100`，只在 response 算 loss。

其餘 —— `BitsAndBytesConfig` 4-bit 配方、`prepare_model_for_kbit_training`、`LoraConfig` / `get_peft_model`、`SFTTrainer` / `SFTConfig`、`save_pretrained` / `merge_and_unload` / `push_to_hub` —— 你在 03/04 全做過，一字未改。這就是 2026 HuggingFace 生態刻意設計的「多模態是單模態的超集」。

### 練習題

1. **遮罩驗證**：把第 4 步 collator 的 image token 也誤設成「不遮罩」，重訓 1 epoch，觀察 loss 曲線與訓練後 VQA 輸出如何崩壞。理解「為什麼一定要遮 image token」。
2. **target_modules 消融**：分別只掛 LLM `q/k/v/o`（不含 projector）、只掛 projector、兩者都掛，比較三組微調後的定性效果與可訓練參數量。
3. **模型可攜性**：把 `MODEL_ID` 換成 `HuggingFaceTB/SmolVLM-Instruct`，確認除了模型常數外幾乎不用改任何程式碼即可跑通，體會 processor 抽象的價值。
4. **merge 精度**：分別在 4-bit base 與 bf16 base 上 `merge_and_unload()`，用同一組測試題比較合併後的輸出差異，驗證第 9 步「要在 bf16 base 上 merge」的論點。
5. **量化階梯對照**：把 4-bit 改成 8-bit（`load_in_8bit=True`）重訓，記錄 VRAM 與訓練後品質，呼應 [`../../04-kbits-tuning`](../../04-kbits-tuning/README.md) 的量化階梯。

### 下一步

- **DPO / 偏好對齊**：本 notebook 做的是 SFT（監督微調）。下一階是用偏好資料對 VLM 做 DPO，讓回答更符合人類偏好、減少幻覺。`trl` 的 `DPOTrainer` 同樣支援多模態，流程心智模型與這裡一致。
- **全參數微調 vs LoRA**：當你有足夠 VRAM 與資料，可比較全參數微調與 LoRA 的效果差距，理解 PEFT 的取捨邊界。
- **整合應用**：把這個微調後的 VLM 接回 [`../05-multimodal_rag/multimodal_embeddings_rag.ipynb`](../05-multimodal_rag/multimodal_embeddings_rag.ipynb) 的多模態 RAG 作為生成端，得到「檢索 + 領域微調生成」的完整系統。

至此，`05-Multimodal` 模組六份 notebook 全部完成。回到 [`../README.md`](../README.md) 重看第 7 節的知識回收對照表，你會發現整個多模態世界，其實都是你在 01–04 既有知識的再應用。